In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

### 1. Generación de Datos (Idéntico a Excel)

In [ ]:
np.random.seed(42) # Garantiza los mismos números aleatorios
n_samples = 50

edad = np.random.normal(40, 10, n_samples).clip(20, 70)
ingreso = np.random.normal(50000, 15000, n_samples).clip(20000, 120000)
deuda_ingreso = np.random.normal(0.3, 0.1, n_samples).clip(0, 0.8)

# Ecuación de riesgo y función sigmoide para determinar el Default
z = -4 + 0.05 * edad - 0.0001 * ingreso + 15 * deuda_ingreso
p_default = 1 / (1 + np.exp(-z))
default = (p_default > 0.5).astype(int)

df = pd.DataFrame({
    'Edad': edad,
    'Ingreso': ingreso,
    'Deuda_Ingreso': deuda_ingreso,
    'Default_Real': default
})

### 2. Preprocesamiento y División (Train/Test)

In [ ]:
# Escalamiento Min-Max
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(df[['Edad', 'Ingreso', 'Deuda_Ingreso']]),
                        columns=['Edad', 'Ingreso', 'Deuda_Ingreso'])
y = df['Default_Real']

# División secuencial 70/30 para replicar las filas exactas del Excel
train_size = int(n_samples * 0.7)
X_train = X_scaled.iloc[:train_size]
y_train = y.iloc[:train_size]

X_test = X_scaled.iloc[train_size:]
y_test = y.iloc[train_size:]

# Exportar a CSV para tener los archivos físicos disponibles
df_train = X_train.copy()
df_train['Default_Real'] = y_train
df_train.to_csv('train_finanzas.csv', index=False)

df_test = X_test.copy()
df_test['Default_Real'] = y_test
df_test.to_csv('test_finanzas.csv', index=False)

print("Archivos CSV generados con éxito.\n")

### 3. Construcción de la Red Neuronal

In [ ]:
# hidden_layer_sizes=(2,) -> 1 capa oculta con 2 neuronas
# activation='logistic' -> Función Sigmoide
# solver='lbfgs' -> Algoritmo cuasi-Newton, similar al GRG Nonlinear de Excel Solver
nn = MLPClassifier(hidden_layer_sizes=(2,), activation='logistic', solver='lbfgs', random_state=42)

# Entrenamiento de la red
nn.fit(X_train, y_train)

### 4. Evaluación y Métricas en Test Set

In [ ]:
# Predecir clases (0 o 1) y probabilidades
y_pred = nn.predict(X_test)
y_prob = nn.predict_proba(X_test)[:, 1] # Probabilidad de Default

print("--- EVALUACIÓN DEL MODELO ---")
print("\n1. Matriz de Confusión (Test Set):")
print(confusion_matrix(y_test, y_pred))

print("\n2. F1-Score y Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

print(f"3. AUC-ROC Score: {roc_auc_score(y_test, y_prob):.4f}")